# 01 — Data Preparation

This notebook prepares the Davis and KIBA datasets.

The main tasks are:

1. Load the raw dataset files
2. Create drug–target pairs
3. Check the data quality
4. Create drug graphs and protein tensors
5. Save the processed datasets

# Imports and project paths

In [2]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

# Find the main project folder
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Allow imports from config.py and src/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import (
    DAVIS_RAW_DIR,
    KIBA_RAW_DIR,
    PROCESSED_DATA_DIR,
    METRICS_DIR,
)

from src.preprocessing import (
    load_graphdta_pairs,
    validate_pairs_dataframe,
    build_drug_graph_cache,
    build_protein_tensor_cache,
    save_pairs_dataframe,
)

from src.utils import (
    set_random_seed,
    create_project_directories,
    save_json,
)

set_random_seed()
create_project_directories()

print("Project folder:", PROJECT_ROOT)
print("Davis folder:", DAVIS_RAW_DIR)
print("KIBA folder:", KIBA_RAW_DIR)

Project folder: /Users/bikashlimbu/Library/CloudStorage/OneDrive2-BuckinghamshireNewUniversity/Dissertation/model/aml_gnn_dti_repurposing
Davis folder: /Users/bikashlimbu/Library/CloudStorage/OneDrive2-BuckinghamshireNewUniversity/Dissertation/model/aml_gnn_dti_repurposing/data/raw/davis
KIBA folder: /Users/bikashlimbu/Library/CloudStorage/OneDrive2-BuckinghamshireNewUniversity/Dissertation/model/aml_gnn_dti_repurposing/data/raw/kiba


## Davis dataset

The Davis dataset contains drug–target binding measurements.

The original Davis values are Kd measurements in nanomolar. They are converted to pKd before model training.

In [4]:
davis_df = load_graphdta_pairs(
    dataset_name="davis",
    raw_data_directory=DAVIS_RAW_DIR,
)

print("Dataset shape:", davis_df.shape)

display(davis_df.head())

Dataset shape: (30056, 8)


,dataset,drug_id,target_id,smiles,protein_sequence,raw_affinity,affinity,affinity_type
0,davis,11314340,AAK1,CC1=C2C=C(C=CC2=NN1)C3=CC(=CN=C3)OCC(CC4=CC=CC...,MKKFFDSRREQGGSGLGSGSSGGGGSTSGLGSGYIGRVFGIGRQQV...,43.0,7.366532,pKd
1,davis,11314340,ABL1(E255K),CC1=C2C=C(C=CC2=NN1)C3=CC(=CN=C3)OCC(CC4=CC=CC...,PFWKILNPLLERGTYYYFMGQQPGKVLGDQRRPSLPALHFIKGAGK...,10000.0,5.000000,pKd
2,davis,11314340,ABL1(F317I),CC1=C2C=C(C=CC2=NN1)C3=CC(=CN=C3)OCC(CC4=CC=CC...,PFWKILNPLLERGTYYYFMGQQPGKVLGDQRRPSLPALHFIKGAGK...,10000.0,5.000000,pKd
3,davis,11314340,ABL1(F317I)p,CC1=C2C=C(C=CC2=NN1)C3=CC(=CN=C3)OCC(CC4=CC=CC...,PFWKILNPLLERGTYYYFMGQQPGKVLGDQRRPSLPALHFIKGAGK...,10000.0,5.000000,pKd
4,davis,11314340,ABL1(F317L),CC1=C2C=C(C=CC2=NN1)C3=CC(=CN=C3)OCC(CC4=CC=CC...,PFWKILNPLLERGTYYYFMGQQPGKVLGDQRRPSLPALHFIKGAGK...,10000.0,5.000000,pKd


In [5]:
print("Number of drug–target pairs:", len(davis_df))
print("Number of unique drugs:", davis_df["drug_id"].nunique())
print("Number of unique targets:", davis_df["target_id"].nunique())
print("Affinity type:", davis_df["affinity_type"].unique())

Number of drug–target pairs: 30056
Number of unique drugs: 68
Number of unique targets: 442
Affinity type: <StringArray>
['pKd']
Length: 1, dtype: str


In [6]:
davis_df["affinity"].describe()

count    30056.000000
mean         5.451535
std          0.894717
min          5.000000
25%          5.000000
50%          5.000000
75%          5.522879
max         10.795880
Name: affinity, dtype: float64

In [7]:
missing_values = davis_df.isnull().sum()

display(
    missing_values.to_frame(
        name="missing_values"
    )
)

,missing_values
dataset,0
drug_id,0
target_id,0
smiles,0
protein_sequence,0
raw_affinity,0
affinity,0
affinity_type,0


In [8]:
duplicate_pairs = davis_df.duplicated(
    subset=["drug_id", "target_id"]
).sum()

print("Duplicate drug–target pairs:", duplicate_pairs)

Duplicate drug–target pairs: 0


In [9]:
davis_validation = validate_pairs_dataframe(
    davis_df
)

validation_table = pd.DataFrame(
    [davis_validation]
)

display(validation_table)

,rows,unique_drugs,unique_targets,duplicate_pairs,invalid_smiles,empty_protein_sequences,non_finite_affinities
0,30056,68,442,0,0,0,0


In [10]:
assert davis_validation["duplicate_pairs"] == 0
assert davis_validation["invalid_smiles"] == 0
assert davis_validation["empty_protein_sequences"] == 0
assert davis_validation["non_finite_affinities"] == 0

print("Davis dataset validation passed.")

Davis dataset validation passed.


## Create drug graphs and protein tensors

Each unique drug is converted from SMILES into a molecular graph.

Each unique protein sequence is converted into a fixed-length numerical tensor.

In [11]:
drug_graph_cache = build_drug_graph_cache(
    davis_df
)

print(
    "Number of drug graphs:",
    len(drug_graph_cache)
)

Number of drug graphs: 68


In [12]:
first_drug_id = str(
    davis_df.iloc[0]["drug_id"]
)

example_graph = drug_graph_cache[
    first_drug_id
]

print("Drug ID:", first_drug_id)
print("Node feature shape:", example_graph.x.shape)
print("Edge shape:", example_graph.edge_index.shape)

Drug ID: 11314340
Node feature shape: torch.Size([27, 6])
Edge shape: torch.Size([2, 60])


In [13]:
protein_tensor_cache = build_protein_tensor_cache(
    davis_df
)

print(
    "Number of protein tensors:",
    len(protein_tensor_cache)
)

Number of protein tensors: 442


In [14]:
first_target_id = str(
    davis_df.iloc[0]["target_id"]
)

example_protein = protein_tensor_cache[
    first_target_id
]

print("Target ID:", first_target_id)
print("Protein tensor shape:", example_protein.shape)
print("First 20 values:", example_protein[:20].tolist())

Target ID: AAK1
Protein tensor shape: torch.Size([1000])
First 20 values: [11, 9, 9, 5, 5, 3, 16, 15, 15, 4, 14, 6, 6, 16, 6, 10, 6, 16, 6, 16]


In [15]:
assert len(drug_graph_cache) == davis_df["drug_id"].nunique()
assert len(protein_tensor_cache) == davis_df["target_id"].nunique()

assert example_graph.x.shape[1] == 6
assert example_protein.shape[0] == 1000

print("Drug graphs and protein tensors created successfully.")

Drug graphs and protein tensors created successfully.


## Save the processed Davis dataset

The processed dataset will be saved as `davis_pairs.csv`.

In [16]:
davis_output_path = (
    PROCESSED_DATA_DIR
    / "davis_pairs.csv"
)

save_pairs_dataframe(
    dataframe=davis_df,
    output_path=davis_output_path,
)

print("Saved to:")
print(davis_output_path)

Saved to:
/Users/bikashlimbu/Library/CloudStorage/OneDrive2-BuckinghamshireNewUniversity/Dissertation/model/aml_gnn_dti_repurposing/data/processed/davis_pairs.csv


In [17]:
validation_output_path = (
    METRICS_DIR
    / "davis_data_validation.json"
)

save_json(
    data=davis_validation,
    output_path=validation_output_path,
)

print("Validation report saved to:")
print(validation_output_path)

Validation report saved to:
/Users/bikashlimbu/Library/CloudStorage/OneDrive2-BuckinghamshireNewUniversity/Dissertation/model/aml_gnn_dti_repurposing/results/metrics/davis_data_validation.json


In [18]:
saved_davis_df = pd.read_csv(
    davis_output_path
)

print("Saved dataset shape:", saved_davis_df.shape)
print("Unique drugs:", saved_davis_df["drug_id"].nunique())
print("Unique targets:", saved_davis_df["target_id"].nunique())

display(saved_davis_df.head())

Saved dataset shape: (30056, 8)
Unique drugs: 68
Unique targets: 442


,dataset,drug_id,target_id,smiles,protein_sequence,raw_affinity,affinity,affinity_type
0,davis,11314340,AAK1,CC1=C2C=C(C=CC2=NN1)C3=CC(=CN=C3)OCC(CC4=CC=CC...,MKKFFDSRREQGGSGLGSGSSGGGGSTSGLGSGYIGRVFGIGRQQV...,43.0,7.366532,pKd
1,davis,11314340,ABL1(E255K),CC1=C2C=C(C=CC2=NN1)C3=CC(=CN=C3)OCC(CC4=CC=CC...,PFWKILNPLLERGTYYYFMGQQPGKVLGDQRRPSLPALHFIKGAGK...,10000.0,5.000000,pKd
2,davis,11314340,ABL1(F317I),CC1=C2C=C(C=CC2=NN1)C3=CC(=CN=C3)OCC(CC4=CC=CC...,PFWKILNPLLERGTYYYFMGQQPGKVLGDQRRPSLPALHFIKGAGK...,10000.0,5.000000,pKd
3,davis,11314340,ABL1(F317I)p,CC1=C2C=C(C=CC2=NN1)C3=CC(=CN=C3)OCC(CC4=CC=CC...,PFWKILNPLLERGTYYYFMGQQPGKVLGDQRRPSLPALHFIKGAGK...,10000.0,5.000000,pKd
4,davis,11314340,ABL1(F317L),CC1=C2C=C(C=CC2=NN1)C3=CC(=CN=C3)OCC(CC4=CC=CC...,PFWKILNPLLERGTYYYFMGQQPGKVLGDQRRPSLPALHFIKGAGK...,10000.0,5.000000,pKd


## Davis data preparation complete

The Davis dataset has been:

- loaded;
- converted to pKd;
- validated;
- converted into drug graphs and protein tensors;
- saved as a processed CSV file.

The next section will prepare the KIBA dataset.

# KIBA Dataset Preparation

The KIBA dataset contains drug–target affinity scores.

Unlike Davis, the KIBA scores are kept in their original form and are not converted to pKd.

In [19]:
kiba_required_files = [
    KIBA_RAW_DIR / "ligands_can.txt",
    KIBA_RAW_DIR / "proteins.txt",
    KIBA_RAW_DIR / "Y",
]

for file_path in kiba_required_files:
    print(file_path.name, "→", file_path.exists())

ligands_can.txt → True
proteins.txt → True
Y → True


In [20]:
kiba_df = load_graphdta_pairs(
    dataset_name="kiba",
    raw_data_directory=KIBA_RAW_DIR,
)

print("Dataset shape:", kiba_df.shape)

display(kiba_df.head())

Dataset shape: (118254, 8)


,dataset,drug_id,target_id,smiles,protein_sequence,raw_affinity,affinity,affinity_type
0,kiba,CHEMBL1087421,O00141,COC1=C(C=C2C(=C1)CCN=C2C3=CC(=C(C=C3)Cl)Cl)Cl,MTVKTEAAKGTLTYSRMRGMVAILIAFMKQRRMGLNDFIQKIANNS...,11.1,11.1,KIBA_score
1,kiba,CHEMBL1087421,O14920,COC1=C(C=C2C(=C1)CCN=C2C3=CC(=C(C=C3)Cl)Cl)Cl,MSWSPSLTTQTCGAWEMKERLGTGGFGNVIRWHNQETGEQIAIKQC...,11.1,11.1,KIBA_score
2,kiba,CHEMBL1087421,O15111,COC1=C(C=C2C(=C1)CCN=C2C3=CC(=C(C=C3)Cl)Cl)Cl,MERPPGLRPGAGGPWEMRERLGTGGFGNVCLYQHRELDLKIAIKSC...,11.1,11.1,KIBA_score
3,kiba,CHEMBL1087421,P00533,COC1=C(C=C2C(=C1)CCN=C2C3=CC(=C(C=C3)Cl)Cl)Cl,MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFED...,11.1,11.1,KIBA_score
4,kiba,CHEMBL1087421,P04626,COC1=C(C=C2C(=C1)CCN=C2C3=CC(=C(C=C3)Cl)Cl)Cl,MELAALCRWGLLLALLPPGAASTQVCTGTDMKLRLPASPETHLDML...,11.1,11.1,KIBA_score


In [22]:
print("Number of drug–target pairs:", len(kiba_df))
print("Number of unique drugs:", kiba_df["drug_id"].nunique())
print("Number of unique targets:", kiba_df["target_id"].nunique())
print("Affinity type:", kiba_df["affinity_type"].unique())

Number of drug–target pairs: 118254
Number of unique drugs: 2111
Number of unique targets: 229
Affinity type: <StringArray>
['KIBA_score']
Length: 1, dtype: str


In [23]:
kiba_df["affinity"].describe()

count    118254.000000
mean         11.719935
std           0.836947
min           0.000000
25%          11.200000
50%          11.500000
75%          11.923909
max          17.200179
Name: affinity, dtype: float64

In [24]:
kiba_missing_values = kiba_df.isnull().sum()

display(
    kiba_missing_values.to_frame(
        name="missing_values"
    )
)

,missing_values
dataset,0
drug_id,0
target_id,0
smiles,0
protein_sequence,0
raw_affinity,0
affinity,0
affinity_type,0


In [25]:
kiba_duplicate_pairs = kiba_df.duplicated(
    subset=["drug_id", "target_id"]
).sum()

print(
    "Duplicate drug–target pairs:",
    kiba_duplicate_pairs
)

Duplicate drug–target pairs: 0


In [26]:
kiba_validation = validate_pairs_dataframe(
    kiba_df
)

display(
    pd.DataFrame([kiba_validation])
)

,rows,unique_drugs,unique_targets,duplicate_pairs,invalid_smiles,empty_protein_sequences,non_finite_affinities
0,118254,2111,229,0,0,0,0


In [27]:
assert kiba_validation["duplicate_pairs"] == 0
assert kiba_validation["invalid_smiles"] == 0
assert kiba_validation["empty_protein_sequences"] == 0
assert kiba_validation["non_finite_affinities"] == 0

print("KIBA dataset validation passed.")

KIBA dataset validation passed.


## Create KIBA drug graphs and protein tensors

Each unique KIBA drug is converted into a molecular graph.

Each unique protein is converted into a fixed-length tensor.

In [28]:
kiba_drug_graph_cache = build_drug_graph_cache(
    kiba_df
)

print(
    "Number of KIBA drug graphs:",
    len(kiba_drug_graph_cache)
)

Number of KIBA drug graphs: 2111


In [29]:
first_kiba_drug_id = str(
    kiba_df.iloc[0]["drug_id"]
)

example_kiba_graph = kiba_drug_graph_cache[
    first_kiba_drug_id
]

print("Drug ID:", first_kiba_drug_id)
print(
    "Node feature shape:",
    example_kiba_graph.x.shape
)
print(
    "Edge shape:",
    example_kiba_graph.edge_index.shape
)

Drug ID: CHEMBL1087421
Node feature shape: torch.Size([21, 6])
Edge shape: torch.Size([2, 46])


In [30]:
kiba_protein_tensor_cache = (
    build_protein_tensor_cache(kiba_df)
)

print(
    "Number of KIBA protein tensors:",
    len(kiba_protein_tensor_cache)
)

Number of KIBA protein tensors: 229


In [31]:
first_kiba_target_id = str(
    kiba_df.iloc[0]["target_id"]
)

example_kiba_protein = (
    kiba_protein_tensor_cache[
        first_kiba_target_id
    ]
)

print("Target ID:", first_kiba_target_id)
print(
    "Protein tensor shape:",
    example_kiba_protein.shape
)

Target ID: O00141
Protein tensor shape: torch.Size([1000])


In [32]:
assert len(kiba_drug_graph_cache) == (
    kiba_df["drug_id"].nunique()
)

assert len(kiba_protein_tensor_cache) == (
    kiba_df["target_id"].nunique()
)

assert example_kiba_graph.x.shape[1] == 6
assert example_kiba_protein.shape[0] == 1000

print(
    "KIBA drug graphs and protein tensors "
    "created successfully."
)

KIBA drug graphs and protein tensors created successfully.


In [33]:
kiba_output_path = (
    PROCESSED_DATA_DIR
    / "kiba_pairs.csv"
)

save_pairs_dataframe(
    dataframe=kiba_df,
    output_path=kiba_output_path,
)

print("KIBA dataset saved to:")
print(kiba_output_path)

KIBA dataset saved to:
/Users/bikashlimbu/Library/CloudStorage/OneDrive2-BuckinghamshireNewUniversity/Dissertation/model/aml_gnn_dti_repurposing/data/processed/kiba_pairs.csv


In [34]:
kiba_validation_output_path = (
    METRICS_DIR
    / "kiba_data_validation.json"
)

save_json(
    data=kiba_validation,
    output_path=kiba_validation_output_path,
)

print("Validation report saved to:")
print(kiba_validation_output_path)

Validation report saved to:
/Users/bikashlimbu/Library/CloudStorage/OneDrive2-BuckinghamshireNewUniversity/Dissertation/model/aml_gnn_dti_repurposing/results/metrics/kiba_data_validation.json


In [35]:
saved_kiba_df = pd.read_csv(
    kiba_output_path
)

print("Saved dataset shape:", saved_kiba_df.shape)
print(
    "Unique drugs:",
    saved_kiba_df["drug_id"].nunique()
)
print(
    "Unique targets:",
    saved_kiba_df["target_id"].nunique()
)

display(saved_kiba_df.head())

Saved dataset shape: (118254, 8)
Unique drugs: 2111
Unique targets: 229


,dataset,drug_id,target_id,smiles,protein_sequence,raw_affinity,affinity,affinity_type
0,kiba,CHEMBL1087421,O00141,COC1=C(C=C2C(=C1)CCN=C2C3=CC(=C(C=C3)Cl)Cl)Cl,MTVKTEAAKGTLTYSRMRGMVAILIAFMKQRRMGLNDFIQKIANNS...,11.1,11.1,KIBA_score
1,kiba,CHEMBL1087421,O14920,COC1=C(C=C2C(=C1)CCN=C2C3=CC(=C(C=C3)Cl)Cl)Cl,MSWSPSLTTQTCGAWEMKERLGTGGFGNVIRWHNQETGEQIAIKQC...,11.1,11.1,KIBA_score
2,kiba,CHEMBL1087421,O15111,COC1=C(C=C2C(=C1)CCN=C2C3=CC(=C(C=C3)Cl)Cl)Cl,MERPPGLRPGAGGPWEMRERLGTGGFGNVCLYQHRELDLKIAIKSC...,11.1,11.1,KIBA_score
3,kiba,CHEMBL1087421,P00533,COC1=C(C=C2C(=C1)CCN=C2C3=CC(=C(C=C3)Cl)Cl)Cl,MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFED...,11.1,11.1,KIBA_score
4,kiba,CHEMBL1087421,P04626,COC1=C(C=C2C(=C1)CCN=C2C3=CC(=C(C=C3)Cl)Cl)Cl,MELAALCRWGLLLALLPPGAASTQVCTGTDMKLRLPASPETHLDML...,11.1,11.1,KIBA_score


# Data Preparation Complete

The Davis and KIBA datasets have been:

- loaded from the raw files;
- checked for missing or invalid data;
- prepared for graph and protein encoding;
- saved as processed CSV files.

The next notebook will train the Davis GCN + CNN baseline.